In [33]:
!pip install openai chromadb langchain-text-splitters langgraph \
             langfuse requests rank_bm25 ragas datasets \
             sentence-transformers -q

In [34]:
import os, json, time, requests
from typing import TypedDict, List, Optional
from openai import OpenAI
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from google.colab import userdata

# ── Grok API (xAI) — OpenAI-compatible, just different base_url + model
os.environ["GROK_API_KEY"] = userdata.get('GROK_API_KEY')   # paste your Grok key here

client = OpenAI(
    api_key=os.environ["GROK_API_KEY"],
    base_url="https://api.x.ai/v1",      # Grok endpoint
)

# ── Free local embeddings (runs on Google's CPU inside Colab)
print("Loading embedding model... (~30 seconds first time)")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Embedding model ready.")

# ── ChromaDB (in-memory for now)
chroma_client = chromadb.Client()

print("All clients ready.")

Loading embedding model... (~30 seconds first time)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model ready.
All clients ready.


In [35]:
def fetch_clinical_trials(condition="lung cancer", max_studies=50):
    """Fetch live trials from ClinicalTrials.gov REST API v2."""
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "query.cond": condition,
        "filter.overallStatus": "RECRUITING,ACTIVE_NOT_RECRUITING",
        "pageSize": min(max_studies, 100),
        "fields": "NCTId,BriefTitle,BriefSummary,Phase,OverallStatus,Condition,InterventionName,LeadSponsorName",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    studies = resp.json().get("studies", [])

    records = []
    for s in studies:
        proto = s.get("protocolSection", {})
        id_mod = proto.get("identificationModule", {})
        desc_mod = proto.get("descriptionModule", {})
        status_mod = proto.get("statusModule", {})
        design_mod = proto.get("designModule", {})
        sponsor_mod = proto.get("sponsorCollaboratorsModule", {})
        arms_mod = proto.get("armsInterventionsModule", {})

        interventions = [i.get("interventionName", "")
                        for i in arms_mod.get("interventions", [])]
        phases = design_mod.get("phases", [])

        records.append({
            "nct_id":        id_mod.get("nctId", ""),
            "title":         id_mod.get("briefTitle", ""),
            "summary":       desc_mod.get("briefSummary", ""),
            "phase":         ", ".join(phases) if phases else "N/A",
            "status":        status_mod.get("overallStatus", ""),
            "condition":     condition,
            "interventions": ", ".join(interventions[:3]),
            "sponsor":       sponsor_mod.get("leadSponsor", {}).get("leadSponsorName", ""),
        })

    print(f"Fetched {len(records)} trials for: {condition}")
    return records

# ── Run it
trials = fetch_clinical_trials(condition="lung cancer", max_studies=50)

# ── Peek at one record
print("\nSample trial:")
print(f"  NCT ID : {trials[0]['nct_id']}")
print(f"  Title  : {trials[0]['title']}")
print(f"  Phase  : {trials[0]['phase']}")
print(f"  Status : {trials[0]['status']}")
print(f"  Sponsor: {trials[0]['sponsor']}")

Fetched 50 trials for: lung cancer

Sample trial:
  NCT ID : NCT07180862
  Title  : A Study Evaluating BAT3306 Compared With Keytruda® in NSCLC Cancer Participants
  Phase  : PHASE1
  Status : RECRUITING
  Sponsor: 


In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)

def ingest_trials(trials, collection_name="clinical_trials"):
    # Create ChromaDB collection
    try:
        chroma_client.delete_collection(collection_name)  # fresh start
    except:
        pass
    collection = chroma_client.create_collection(collection_name)

    all_chunks, all_ids, all_metadata = [], [], []

    for trial in trials:
        # Combine all text fields into one searchable block
        text = f"""Title: {trial['title']}
Phase: {trial['phase']}
Status: {trial['status']}
Condition: {trial['condition']}
Interventions: {trial['interventions']}
Summary: {trial['summary']}"""

        chunks = splitter.split_text(text)

        for j, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_ids.append(f"{trial['nct_id']}_chunk_{j}")
            all_metadata.append({
                "nct_id":    trial['nct_id'],
                "title":     trial['title'][:80],
                "phase":     trial['phase'],
                "condition": trial['condition'],
                "source":    "clinicaltrials.gov",
            })

    # Embed using local sentence-transformers (free)
    print(f"Embedding {len(all_chunks)} chunks locally...")
    embeddings = embed_model.encode(all_chunks, show_progress_bar=True).tolist()

    # Store in ChromaDB
    collection.add(
        ids=all_ids,
        documents=all_chunks,
        embeddings=embeddings,
        metadatas=all_metadata,
    )

    print(f"\nStored {len(all_chunks)} chunks in ChromaDB.")
    print(f"Collection '{collection_name}' is ready to search.")
    return collection

# ── Run it
collection = ingest_trials(trials)

Embedding 118 chunks locally...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Stored 118 chunks in ChromaDB.
Collection 'clinical_trials' is ready to search.


In [37]:
def search(query, collection, n_results=5):
    """Convert query to vector, find closest chunks in ChromaDB."""

    # Embed the query using the same model we used for storage
    query_vector = embed_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_vector,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text":     doc,
            "nct_id":   meta.get("nct_id", ""),
            "title":    meta.get("title", ""),
            "phase":    meta.get("phase", ""),
            "score":    round(1 - dist, 3),   # distance → similarity
        })

    return retrieved

# ── Test with 3 queries
test_queries = [
    "EGFR targeted therapy trials",
    "immunotherapy combination treatment",
    "Phase 2 recruiting trials",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = search(query, collection, n_results=3)
    for r in results:
        print(f"  [{r['score']}] {r['nct_id']} — {r['title']}")


Query: 'EGFR targeted therapy trials'
  [0.548] NCT06385483 — Testing Afatinib as Potentially Targeted Treatment in Cancers With EGFR Genetic 
  [0.522] NCT06418412 — A Pan-Asian Clinical Database of EGFR Exon 20 Insertion Mutated NSCLC
  [0.504] NCT03774732 — PD-1 Inhibitor and Chemotherapy With Concurrent Irradiation at Varied Tumour Sit

Query: 'immunotherapy combination treatment'
  [0.496] NCT04013542 — Ipilimumab and Nivolumab in Combination With Radiation Therapy in Treating Patie
  [0.468] NCT06896422 — Randomized Trial of Glutathione With Anti-PD-1 and Chemotherapy in Advanced NSCL
  [0.468] NCT06465329 — A Study of Cemiplimab Plus Chemotherapy Versus Cemiplimab Plus Chemotherapy Plus

Query: 'Phase 2 recruiting trials'
  [0.432] NCT07070518 — Study of GV20-0251 in Participants With Solid Tumor Malignancies
  [0.392] NCT06385483 — Testing Afatinib as Potentially Targeted Treatment in Cancers With EGFR Genetic 
  [0.387] NCT01696994 — Screening for Ovarian Cancer in Older Pati

In [38]:
# ── Agent State — shared memory passed between all agents
class ClinicalState(TypedDict):
    query:            str
    intent:           str        # "trials" | "evidence" | "safety" | "hybrid"
    trial_results:    List[dict]
    citations:        List[str]
    final_answer:     str
    error:            Optional[str]

# ── Supervisor — reads the question, decides which agent handles it
def supervisor(state: ClinicalState) -> ClinicalState:
    prompt = f"""You are routing a clinical research question to the right specialist.

Classify this query into exactly one of these categories:
- trials     → user wants to find clinical trials
- evidence   → user wants research literature or study findings
- safety     → user wants adverse events or drug safety data
- hybrid     → user needs trials AND evidence combined

Query: {state['query']}

Reply with only the single category word."""

    resp = client.chat.completions.create(
        model="grok-3-mini",       # cheap Grok model for routing
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
    )

    intent = resp.choices[0].message.content.strip().lower()
    if intent not in ["trials", "evidence", "safety", "hybrid"]:
        intent = "hybrid"   # safe default

    state["intent"] = intent
    print(f"  Supervisor → intent: '{intent}'")
    return state

# ── Test the supervisor alone
test_state: ClinicalState = {
    "query":         "What Phase 2 trials are recruiting for EGFR lung cancer?",
    "intent":        "",
    "trial_results": [],
    "citations":     [],
    "final_answer":  "",
    "error":         None,
}

result = supervisor(test_state)
print(f"Query classified as: {result['intent']}")

  Supervisor → intent: 'trials'
Query classified as: trials


In [39]:
def trial_agent(state: ClinicalState) -> ClinicalState:
    """Searches ChromaDB for relevant trials and adds them to state."""

    print(f"  Trial Agent → searching for: '{state['query']}'")

    results = search(state["query"], collection, n_results=5)

    # Store results + citations in shared state
    seen = set()
    unique_results = []
    for r in results:
        if r["nct_id"] not in seen:
            seen.add(r["nct_id"])
            unique_results.append(r)

    state["trial_results"] = unique_results
    state["citations"] = [r["nct_id"] for r in unique_results]

    print(f"  Trial Agent → found {len(results)} trials")
    print(f"  Citations: {', '.join(state['citations'])}")

    return state

# ── Test the trial agent alone
test_state["intent"] = "trials"   # set from supervisor

result = trial_agent(test_state)

print("\nTop trial found:")
print(f"  {result['trial_results'][0]['nct_id']} — {result['trial_results'][0]['title']}")
print(f"  Phase: {result['trial_results'][0]['phase']}")
print(f"  Score: {result['trial_results'][0]['score']}")

  Trial Agent → searching for: 'What Phase 2 trials are recruiting for EGFR lung cancer?'
  Trial Agent → found 5 trials
  Citations: NCT06385483, NCT07070518, NCT06896422, NCT06257264, NCT03793179

Top trial found:
  NCT06385483 — Testing Afatinib as Potentially Targeted Treatment in Cancers With EGFR Genetic 
  Phase: PHASE2
  Score: 0.624


In [40]:
def synthesis_agent(state: ClinicalState) -> ClinicalState:
    """Reads trial results from state, calls Grok to write a grounded answer."""

    print(f"  Synthesis Agent → generating answer...")

    # Build context from retrieved trials
    context_parts = []
    for trial in state["trial_results"][:3]:
        context_parts.append(
            f"Trial {trial['nct_id']}: {trial['title']}\n"
            f"Phase: {trial['phase']} | Score: {trial['score']}\n"
            f"Excerpt: {trial['text'][:200]}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""You are a clinical research assistant helping a biopharma scientist.

Answer the question using ONLY the trials listed below.
- Cite specific NCT IDs (e.g. NCT06385483) in your answer
- If a trial is not relevant, skip it
- Do not add information not in the context
- Be concise — 3 to 5 sentences

CONTEXT:
{context}

QUESTION: {state['query']}

Answer:"""

    resp = client.chat.completions.create(
        model="grok-3-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )

    state["final_answer"] = resp.choices[0].message.content.strip()
    print("  Synthesis Agent → answer ready.")
    return state

# ── Test it
result = synthesis_agent(test_state)

print("\n" + "="*60)
print("QUESTION:", test_state["query"])
print("="*60)
print(result["final_answer"])
print("="*60)
print("CITATIONS:", ", ".join(result["citations"]))

  Synthesis Agent → generating answer...
  Synthesis Agent → answer ready.

QUESTION: What Phase 2 trials are recruiting for EGFR lung cancer?
No Phase 2 trials in the provided context are both recruiting and targeted to EGFR lung cancer. NCT06385483 is a Phase 2 trial for EGFR genetic changes in lung cancer (MATCH - Subprotocol A) but has status ACTIVE_NOT_RECRUITING. NCT07070518 includes a Phase 2 component and is recruiting for lung cancer, but the excerpt does not specify EGFR. NCT06896422 is limited to Phase 1.
CITATIONS: NCT06385483, NCT07070518, NCT06896422, NCT06257264, NCT03793179


In [41]:
# Ingest 3 more disease conditions to expand coverage
more_conditions = [
    "non-small cell lung cancer",
    "KRAS mutation cancer",
    "EGFR mutation",
]

# Store trials along with the condition they were fetched under
trials_with_conditions = []
for query_condition in more_conditions:
    new_trials = fetch_clinical_trials(condition=query_condition, max_studies=50)
    for trial in new_trials:
        trials_with_conditions.append((trial, query_condition))
    time.sleep(1) # be polite to the API, apply after each fetch

print(f"\nTotal new trials fetched: {len(trials_with_conditions)}")

# Add to existing collection
all_chunks, all_ids, all_metadata = [], [], []

for trial, query_condition in trials_with_conditions:
    text = f"""Title: {trial['title']}
Phase: {trial['phase']}
Status: {trial['status']}
Condition: {trial['condition']}
Interventions: {trial['interventions']}
Summary: {trial['summary']}"""

    chunks = splitter.split_text(text)
    for j, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        # Ensure the ID includes the specific query_condition under which this trial was fetched
        all_ids.append(f"{trial['nct_id']}_{query_condition[:10]}_chunk_{j}")
        all_metadata.append({
            "nct_id":    trial['nct_id'],
            "title":     trial['title'][:80],
            "phase":     trial['phase'],
            "condition": trial['condition'], # This is the trial's official condition, useful for metadata
            "source":    "clinicaltrials.gov",
        })

print(f"Embedding {len(all_chunks)} new chunks...")
new_embeddings = embed_model.encode(all_chunks, show_progress_bar=True).tolist()

collection.upsert(
    ids=all_ids,
    documents=all_chunks,
    embeddings=new_embeddings,
    metadatas=all_metadata,
)

print(f"Collection now has richer coverage. Re-run your queries!")

Fetched 50 trials for: non-small cell lung cancer
Fetched 50 trials for: KRAS mutation cancer
Fetched 50 trials for: EGFR mutation

Total new trials fetched: 150
Embedding 317 new chunks...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Collection now has richer coverage. Re-run your queries!


In [42]:
def run_pipeline(query: str, drug_name: str = "") -> dict:
    """Runs a query through all agents end-to-end."""

    print(f"\nQuery: {query}")
    print("-" * 50)

    # Build fresh state
    state: ClinicalState = {
        "query":         query,
        "intent":        "",
        "trial_results": [],
        "citations":     [],
        "final_answer":  "",
        "error":         None,
    }

    # Run agents in sequence
    state = supervisor(state)
    state = trial_agent(state)
    state = synthesis_agent(state)

    if drug_name:                                    # only if drug specified
        state = risk_agent(state, drug_name)

    return {
        "question": state["query"],
        "intent":   state["intent"],
        "answer":   state["final_answer"],
        "citations": state["citations"],
    }

# ── Run 3 real questions
questions = [
    "What Phase 2 trials are recruiting for EGFR lung cancer?",
    "Are there any immunotherapy combination trials open for NSCLC?",
    "Which trials are studying targeted therapy for lung cancer mutations?",
]

for q in questions:
    result = run_pipeline(q)
    print("\nANSWER:", result["answer"])
    print("CITATIONS:", ", ".join(result["citations"][:3]))
    print("=" * 50)


Query: What Phase 2 trials are recruiting for EGFR lung cancer?
--------------------------------------------------
  Supervisor → intent: 'trials'
  Trial Agent → searching for: 'What Phase 2 trials are recruiting for EGFR lung cancer?'
  Trial Agent → found 5 trials
  Citations: NCT06385483, NCT07070518, NCT06015503
  Synthesis Agent → generating answer...
  Synthesis Agent → answer ready.

ANSWER: No Phase 2 trials listed are recruiting for EGFR lung cancer. NCT06385483 is a Phase 2 trial for EGFR genetic changes in lung cancer but has status ACTIVE_NOT_RECRUITING. NCT06015503 is a Phase 2 trial for EGFR ex20ins mutation in NSCLC but has status ACTIVE_NOT_RE. NCT07070518 is recruiting (Phase 1/2) for lung cancer but does not reference EGFR.
CITATIONS: NCT06385483, NCT07070518, NCT06015503

Query: Are there any immunotherapy combination trials open for NSCLC?
--------------------------------------------------
  Supervisor → intent: 'trials'
  Trial Agent → searching for: 'Are there a

In [43]:
def query_faers(drug_name: str) -> dict:
    """Query FDA's public adverse events database. No API key needed."""
    url = "https://api.fda.gov/drug/event.json"
    params = {
        "search": f'patient.drug.medicinalproduct:"{drug_name}"',
        "count":  "patient.reaction.reactionmeddrapt.exact",
        "limit":  8,
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        return {
            "drug":   drug_name,
            "events": [{"reaction": r["term"], "count": r["count"]}
                       for r in results],
        }
    except Exception as e:
        return {"drug": drug_name, "events": [], "error": str(e)}


def risk_agent(state: ClinicalState, drug_name: str) -> ClinicalState:
    """Queries FDA FAERS for the drug in question and adds safety context."""

    print(f"  Risk Agent → querying FDA FAERS for: '{drug_name}'")
    faers = query_faers(drug_name)

    if not faers["events"]:
        state["final_answer"] += "\n\nNo FDA adverse event data found for this drug."
        return state

    # Format top adverse events
    ae_lines = "\n".join(
        [f"  - {e['reaction']}: {e['count']:,} reports"
         for e in faers["events"][:5]]
    )

    # Ask Grok to add a safety summary paragraph
    prompt = f"""Add a brief safety note (2 sentences max) to this clinical answer.

Existing answer:
{state['final_answer']}

FDA adverse event data for {drug_name}:
{ae_lines}

Append a sentence starting with: "FDA adverse event data shows..." """

    resp = client.chat.completions.create(
        model="grok-3-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120,
    )

    safety_note = resp.choices[0].message.content.strip()
    state["final_answer"] = state["final_answer"] + "\n\n" + safety_note
    print(f"  Risk Agent → safety note added.")
    return state


# ── Test: run full pipeline then add safety layer
result_state: ClinicalState = {
    "query":         "What trials are studying osimertinib for EGFR lung cancer?",
    "intent":        "",
    "trial_results": [],
    "citations":     [],
    "final_answer":  "",
    "error":         None,
}

result_state = supervisor(result_state)
result_state = trial_agent(result_state)
result_state = synthesis_agent(result_state)
result_state = risk_agent(result_state, drug_name="osimertinib")

print("\n" + "="*60)
print("FINAL ANSWER WITH SAFETY DATA:")
print("="*60)
print(result_state["final_answer"])
print("\nCITATIONS:", ", ".join(result_state["citations"]))

  Supervisor → intent: 'trials'
  Trial Agent → searching for: 'What trials are studying osimertinib for EGFR lung cancer?'
  Trial Agent → found 5 trials
  Citations: NCT07058519, NCT05801029, NCT03521154, NCT07505173
  Synthesis Agent → generating answer...
  Synthesis Agent → answer ready.
  Risk Agent → querying FDA FAERS for: 'osimertinib'
  Risk Agent → safety note added.

FINAL ANSWER WITH SAFETY DATA:
Trials NCT07058519 and NCT05801029 are studying osimertinib for EGFR lung cancer. NCT07058519 is a phase 2 trial of osimertinib-based adaptive treatment guided by ctDNA EGFRm+ monitoring in NSCLC. NCT05801029 is a phase 2 trial of osimertinib and amivantamab in NSCLC with common EGFR mutations.

Trials NCT07058519 and NCT05801029 are studying osimertinib for EGFR lung cancer. NCT07058519 is a phase 2 trial of osimertinib-based adaptive treatment guided by ctDNA EGFRm+ monitoring in NSCLC. NCT05801029 is a phase 2 trial of osimertinib and amivantamab in NSCLC with common EGFR mutat

In [44]:
!pip install langchain-openai langchain-community -q

In [45]:
!pip install langchain-google-vertexai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 675.2/675.2 kB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires langchain-core<0.3.0,>=0.2.43, but you have langchain-core 1.5.1 which is incompatible.
langchain 0.2.17 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.10.10 which is incompatible.
ragas 0.1.21 requires langchain-core<0.3, but you have langchain-core 1.5.1 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph-prebuilt 0.1.8 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.15,!=0.3.16,!=0.3.17,!=0.3.18,!=0.3.19,!=0.3.2,!=0.3.20,!=0.3.21,!=0.3.22,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-

In [46]:
!pip install google-cloud-aiplatform -q

In [47]:
!pip install ragas==0.1.21 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-google-vertexai 3.2.4 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.2.43 which is incompatible.


In [48]:
def evaluate_faithfulness(answer: str, contexts: list[str]) -> float:
    """
    Checks each sentence in the answer against retrieved context.
    Faithfulness = sentences grounded in context / total sentences
    This is exactly what RAGAS does — just without the dependency.
    """
    context_block = "\n".join(contexts)

    # Split answer into sentences
    sentences = [s.strip() for s in answer.replace("\n", " ").split(".")
                 if len(s.strip()) > 20]

    if not sentences:
        return 0.0

    grounded = 0
    for sentence in sentences:
        prompt = f"""Context:
{context_block[:1500]}

Claim: "{sentence}"

Is this claim directly supported by the context above?
Reply with only YES or NO."""

        resp = client.chat.completions.create(
            model="grok-3-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5,
        )
        verdict = resp.choices[0].message.content.strip().upper()
        if verdict == "YES":
            grounded += 1

    return round(grounded / len(sentences), 3)


def evaluate_relevancy(question: str, answer: str) -> float:
    """Checks if the answer actually addresses the question."""
    prompt = f"""Question: {question}
Answer: {answer}

On a scale from 0.0 to 1.0, how well does the answer address the question?
Consider: does it answer what was asked? Are the citations relevant?
Reply with only a decimal number like 0.85"""

    resp = client.chat.completions.create(
        model="grok-3-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
    )
    try:
        return float(resp.choices[0].message.content.strip())
    except:
        return 0.0


print("Evaluator functions ready.")

Evaluator functions ready.


In [49]:
eval_questions = [
    "What Phase 2 trials are studying osimertinib for EGFR lung cancer?",
    "Are there immunotherapy combination trials recruiting for NSCLC?",
    "Which trials are studying targeted therapy for lung cancer mutations?",
    "What trials focus on KRAS mutation treatment in lung cancer?",
    "What trials are studying amivantamab for lung cancer?",
]

print("Running evaluation...\n")
faithfulness_scores = []
relevancy_scores = []

for question in eval_questions:
    # Run pipeline
    state: ClinicalState = {
        "query": question, "intent": "", "trial_results": [],
        "citations": [], "final_answer": "", "error": None,
    }
    state = supervisor(state)
    state = trial_agent(state)
    state = synthesis_agent(state)

    contexts = [
    f"Trial {r['nct_id']}: {r['title']}\nPhase: {r['phase']}\nExcerpt: {r['text'][:300]}"
    for r in state["trial_results"]
]
    answer   = state["final_answer"]

    f_score = evaluate_faithfulness(answer, contexts)
    r_score = evaluate_relevancy(question, answer)

    faithfulness_scores.append(f_score)
    relevancy_scores.append(r_score)

    print(f"Q: {question[:55]}...")
    print(f"   Faithfulness: {f_score}  |  Relevancy: {r_score}\n")

avg_f = round(sum(faithfulness_scores) / len(faithfulness_scores), 3)
avg_r = round(sum(relevancy_scores) / len(relevancy_scores), 3)

print("="*45)
print("   EVALUATION SUMMARY")
print("="*45)
print(f"   Avg Faithfulness:  {avg_f}  (target >0.90)")
print(f"   Avg Relevancy:     {avg_r}  (target >0.85)")
print("="*45)
print("\nThese are your FDE interview numbers.")

Running evaluation...

  Supervisor → intent: 'trials'
  Trial Agent → searching for: 'What Phase 2 trials are studying osimertinib for EGFR lung cancer?'
  Trial Agent → found 5 trials
  Citations: NCT07058519, NCT05801029, NCT07505173, NCT07375316
  Synthesis Agent → generating answer...
  Synthesis Agent → answer ready.
Q: What Phase 2 trials are studying osimertinib for EGFR l...
   Faithfulness: 0.75  |  Relevancy: 0.9

  Supervisor → intent: 'trials'
  Trial Agent → searching for: 'Are there immunotherapy combination trials recruiting for NSCLC?'
  Trial Agent → found 5 trials
  Citations: NCT07014202, NCT06849167, NCT07008742, NCT06634199
  Synthesis Agent → generating answer...
  Synthesis Agent → answer ready.
Q: Are there immunotherapy combination trials recruiting f...
   Faithfulness: 1.0  |  Relevancy: 0.9

  Supervisor → intent: 'trials'
  Trial Agent → searching for: 'Which trials are studying targeted therapy for lung cancer mutations?'
  Trial Agent → found 5 trials
  